# Multi-faceted Filtering for Improved Retrieval in RAG Systems

## Overview
Multi-faceted filtering refines the set of retrieved documents by applying several complementary filters before the results reach the LLM. Each filter targets a different quality dimension of the retrieved results:

- **Metadata Filtering 🔖**: keep only documents whose metadata matches the query's constraints (e.g., country, company, or date).
- **Similarity Thresholding 📏**: drop documents whose relevance score falls below a threshold.
- **Content Filtering 🔑**: require essential keywords to appear in the document content.
- **Diversity Filtering 🌈**: remove near-duplicate documents so the final context covers more distinct information.

Applying these filters together produces a smaller, higher-quality context, which reduces noise, token cost and hallucination risk in the generated answer.

The notebook demonstrates the technique on a customer dataset with rich metadata, using LangChain and a Chroma vector store.

<div style="text-align: center;">

<img src="../images/multi_faceted_filtering.svg" alt="Multi-faceted Filtering pipeline" style="width:100%; max-width:1000px;">

</div>

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.

In [ ]:
# Install required packages
!pip install langchain langchain-chroma langchain-huggingface langchain-openai python-dotenv sentence-transformers

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git

In [ ]:
import csv
import os
import sys
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

sys.path.append(os.path.abspath('RAG_TECHNIQUES'))
from helper_functions import text_wrap

# Load environment variables from a .env file
load_dotenv()

### Download the dataset

The notebook uses `customers-100.csv`, a small customer dataset included in the repository. Each row becomes one `Document` whose metadata keeps the customer's country, company, city, subscription date and website — ideal for demonstrating metadata-aware filtering.

In [ ]:
# Download required data files
os.makedirs('data', exist_ok=True)

# Download the customer dataset used in this notebook
!wget -O data/customers-100.csv https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/customers-100.csv

In [ ]:
def load_customer_documents(csv_path: str) -> List[Document]:
    """Convert each row of the customer dataset into a Document with rich metadata."""
    docs = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            content = (
                f"{row['First Name']} {row['Last Name']} from {row['Company']} "
                f"in {row['City']}, {row['Country']}. "
                f"Subscribed on {row['Subscription Date']}. "
                f"Website: {row['Website']}"
            )
            docs.append(Document(
                page_content=content,
                metadata={
                    "country": row["Country"],
                    "company": row["Company"],
                    "city": row["City"],
                    "subscription_date": row["Subscription Date"],
                    "website": row["Website"],
                },
            ))
    return docs


customers = load_customer_documents("data/customers-100.csv")
print(f"Loaded {len(customers)} customer documents")
print(customers[0].page_content)
print(customers[0].metadata)

### Build the vector store

Embed every customer document with a local sentence-transformer model and index the embeddings in a Chroma vector store.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(customers, embedding=embeddings)

### Baseline retrieval

Retrieve the top-10 documents for a query. Notice that the raw result set mixes relevant and less relevant documents, and that several results can carry overlapping information.

In [ ]:
query = "Which customers work for companies in Chile?"

results = vectorstore.similarity_search_with_relevance_scores(query, k=10)
print(f"Retrieved {len(results)} documents\n")
for doc, score in results:
    print(f"{score:.3f} | {doc.metadata['country']:<10} | {doc.metadata['company']}")

## Filter 1: Metadata Filtering 🔖

Metadata filtering keeps only the documents whose metadata matches the query's constraints. In this example we are only interested in customers from Chile, so we filter on the `country` field.

In [ ]:
def apply_metadata_filter(
    doc_score_pairs: List[Tuple[Document, float]],
    filters: Optional[Dict[str, Any]] = None,
) -> List[Tuple[Document, float]]:
    """Keep only documents whose metadata matches the given key-value filters."""
    if not filters:
        return doc_score_pairs
    return [
        (doc, score) for doc, score in doc_score_pairs
        if all(doc.metadata.get(key) == value for key, value in filters.items())
    ]


metadata_filtered = apply_metadata_filter(results, {"country": "Chile"})
print(f"Kept {len(metadata_filtered)} of {len(results)} documents\n")
for doc, score in metadata_filtered:
    print(f"{score:.3f} | {doc.metadata['city']:<15} | {doc.metadata['company']}")

## Filter 2: Similarity Thresholding 📏

Similarity thresholding drops documents whose relevance score is below a threshold, keeping only the most pertinent results. The threshold should be tuned per embedding model and corpus.

In [ ]:
def apply_similarity_threshold(
    doc_score_pairs: List[Tuple[Document, float]],
    threshold: float = 0.35,
) -> List[Tuple[Document, float]]:
    """Keep only documents whose relevance score is above the threshold."""
    return [(doc, score) for doc, score in doc_score_pairs if score >= threshold]


threshold_filtered = apply_similarity_threshold(results, threshold=0.35)
print(f"Kept {len(threshold_filtered)} of {len(results)} documents\n")
for doc, score in threshold_filtered:
    print(f"{score:.3f} | {doc.metadata['country']:<10} | {doc.metadata['company']}")

## Filter 3: Content Filtering 🔑

Content filtering keeps only documents whose text contains the required keywords. This is useful when the query implies terms that must appear in the answer's evidence.

In [ ]:
def apply_content_filter(
    doc_score_pairs: List[Tuple[Document, float]],
    keywords: Optional[List[str]] = None,
    require_all: bool = True,
) -> List[Tuple[Document, float]]:
    """Keep only documents whose content contains the required keywords.

    With ``require_all=True`` every keyword must appear in the document;
    otherwise a single match is enough.
    """
    if not keywords:
        return doc_score_pairs
    keywords_lower = [k.lower() for k in keywords]
    filtered = []
    for doc, score in doc_score_pairs:
        content = doc.page_content.lower()
        matches = [k in content for k in keywords_lower]
        if (all(matches) if require_all else any(matches)):
            filtered.append((doc, score))
    return filtered


content_filtered = apply_content_filter(results, ["chile"])
print(f"Kept {len(content_filtered)} of {len(results)} documents\n")
for doc, score in content_filtered:
    print(f"{score:.3f} | {doc.metadata['country']:<10} | {doc.metadata['company']}")

## Filter 4: Diversity Filtering 🌈

Diversity filtering removes near-duplicate documents so the final context covers more distinct information. The implementation below is a greedy MMR-style pass: documents are kept only if their cosine similarity to every already-kept document is below a threshold.

In [ ]:
def apply_diversity_filter(
    doc_score_pairs: List[Tuple[Document, float]],
    embeddings: HuggingFaceEmbeddings,
    similarity_threshold: float = 0.9,
) -> List[Tuple[Document, float]]:
    """Remove near-duplicate documents greedily using embedding cosine similarity."""
    if embeddings is None:
        raise ValueError(
            "embeddings are required when diversity filtering is enabled; "
            "pass a HuggingFaceEmbeddings instance to apply_diversity_filter"
        )
    kept: List[Tuple[Document, float]] = []
    kept_vectors: List[np.ndarray] = []
    for doc, score in doc_score_pairs:
        vector = np.asarray(embeddings.embed_query(doc.page_content))
        is_duplicate = any(
            float(np.dot(vector, other) / (np.linalg.norm(vector) * np.linalg.norm(other)))
            >= similarity_threshold
            for other in kept_vectors
        )
        if not is_duplicate:
            kept.append((doc, score))
            kept_vectors.append(vector)
    return kept


diversity_filtered = apply_diversity_filter(results, embeddings, similarity_threshold=0.9)
print(f"Kept {len(diversity_filtered)} of {len(results)} documents\n")
for doc, score in diversity_filtered:
    print(f"{score:.3f} | {doc.metadata['country']:<10} | {doc.metadata['company']}")

## Putting It All Together 🧩

The four filters compose into a single pipeline. Each filter is optional, so the pipeline can be adapted to the retrieval scenario at hand.

In [ ]:
def multi_faceted_filter(
    doc_score_pairs: List[Tuple[Document, float]],
    metadata_filters: Optional[Dict[str, Any]] = None,
    score_threshold: Optional[float] = None,
    required_keywords: Optional[List[str]] = None,
    diversity_threshold: Optional[float] = None,
    embeddings: Optional[HuggingFaceEmbeddings] = None,
) -> List[Tuple[Document, float]]:
    """Apply metadata, similarity, content and diversity filters in sequence."""
    filtered = doc_score_pairs
    if metadata_filters:
        filtered = apply_metadata_filter(filtered, metadata_filters)
    if score_threshold is not None:
        filtered = apply_similarity_threshold(filtered, score_threshold)
    if required_keywords:
        filtered = apply_content_filter(filtered, required_keywords)
    if diversity_threshold is not None:
        filtered = apply_diversity_filter(filtered, embeddings, diversity_threshold)
    return filtered


filtered = multi_faceted_filter(
    results,
    metadata_filters={"country": "Chile"},
    score_threshold=0.35,
    required_keywords=["chile"],
    diversity_threshold=0.9,
    embeddings=embeddings,
)

print(f"Before filtering: {len(results)} documents")
print(f"After filtering:  {len(filtered)} documents\n")
for doc, score in filtered:
    print(f"{score:.3f} | {doc.metadata['city']:<15} | {doc.metadata['company']}")

### Use Case example

End-to-end example: retrieve candidates, filter them with the full pipeline, and generate an answer from the clean context.

In [ ]:
query = "Which customer accounts in Chile should our sales team prioritize, and which company does each one belong to?"

# Retrieve candidates and apply the full filtering pipeline
results = vectorstore.similarity_search_with_relevance_scores(query, k=10)
filtered = multi_faceted_filter(
    results,
    metadata_filters={"country": "Chile"},
    score_threshold=0.35,
    required_keywords=["chile"],
    diversity_threshold=0.9,
    embeddings=embeddings,
)

context = "\n\n".join(text_wrap(doc.page_content) for doc, _ in filtered)

llm = ChatOpenAI(model="gpt-4o", temperature=0)
response = llm.invoke(
    "Based on the following customer records, answer the question.\n\n"
    f"Question: {query}\n\nRecords:\n{context}"
)
print(response.content)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--multi-faceted-filtering)